# Large Benchmark Data

## Packages

In [ ]:
import torch
import gpytorch
import numpy as np
import ctypes
import gpboost as gpb
import requests
import pandas as pd
import time

## Data

In [ ]:
# Load the solution data (Switch between 2a and 2b)
data_solution = pd.read_csv(
    "https://raw.githubusercontent.com/TimGyger/SpaceTimeGPApprox/refs/heads/main/Data/2a/2a-solutions.csv"
)

In [ ]:
# Initialize result vectors
vec_vecchia_RMSE = np.zeros(9)
vec_vecchia_corr_RMSE = np.zeros(9)
vec_fitc_RMSE = np.zeros(9)
vec_fitc_grid_RMSE = np.zeros(9)
vec_vif_RMSE = np.zeros(9)

vec_vecchia_time = np.zeros(9)
vec_vecchia_corr_time = np.zeros(9)
vec_fitc_time = np.zeros(9)
vec_fitc_grid_time = np.zeros(9)
vec_vif_time = np.zeros(9)

## Experiments

In [ ]:
for i in range(0, 10):
    # Load training and test data (Switch between 2a and 2b)
    data_train = pd.read_csv(
        f"https://raw.githubusercontent.com/TimGyger/SpaceTimeGPApprox/refs/heads/main/Data/2a/2a_{i+1}_train.csv"
    )
    data_test = pd.read_csv(
        f"https://raw.githubusercontent.com/TimGyger/SpaceTimeGPApprox/refs/heads/main/Data/2a/2a_{i+1}_test.csv"
    )

    coords_train = data_train[["t", "x", "y"]].to_numpy()
    y_train = data_train["z"].to_numpy()
    coords_test = data_test[["t", "x", "y"]].to_numpy()
    y_test = data_solution[f"z{i+1}"].to_numpy()

    cov_pars = [0.00001, 1, 0.5, 20, 0.5, 1.5, 0.9, 0.1]

    # Vecchia (Euclidean)
    start_time = time.time()
    model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="gaussian",
                            vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "cholesky", gp_approx="vecchia",num_neighbors = 30)
    model.fit(y = y_train,params={"trace": True, "estimate_cov_par_index": [1,1,1,1,1,1,1,1],
                                  "init_cov_pars": cov_pars})

    pred_Linear_Model = model.predict(gp_coords_pred=coords_test,y=y_train) 
    end_time = time.time()
    vec_vecchia_RMSE[i-1] = np.sqrt(np.mean((y_test-pred_Linear_Model["mu"])**2))
    vec_vecchia_time[i-1] = end_time-start_time
    print("Vecchia RMSE:", vec_vecchia_RMSE[i-1], "Time:", vec_vecchia_time[i-1])

    # Vecchia (Correlation)
    start_time = time.time()
    model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="gaussian",
                            vecchia_ordering = "time_random_space",
                            matrix_inversion_method = "cholesky", gp_approx="vecchia_correlation_based",num_neighbors = 30)
    model.fit(y = y_train,params={"trace": True, "estimate_cov_par_index": [1,1,1,1,1,1,1,1],
                                  "init_cov_pars": cov_pars})

    pred_Linear_Model = model.predict(gp_coords_pred=coords_test,y=y_train) 
    end_time = time.time()
    vec_vecchia_corr_RMSE[i-1] = np.sqrt(np.mean((y_test-pred_Linear_Model["mu"])**2))
    vec_vecchia_corr_time[i-1] = end_time-start_time
    print("Vecchia Corr RMSE:", vec_vecchia_corr_RMSE[i-1], "Time:", vec_vecchia_corr_time[i-1])

    # FITC (kmeans++)
    start_time = time.time()
    model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="gaussian",num_ind_points = 500,ind_points_selection = "kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
    model.fit(y = y_train,params={"trace": True, "estimate_cov_par_index": [1,1,1,1,1,1,1,1],
                                  "init_cov_pars": cov_pars})

    pred_Linear_Model = model.predict(gp_coords_pred=coords_test,y=y_train) 
    end_time = time.time()
    vec_fitc_RMSE[i-1] = np.sqrt(np.mean((y_test-pred_Linear_Model["mu"])**2))
    vec_fitc_time[i-1] = end_time-start_time
    print("FITC RMSE:", vec_fitc_RMSE[i-1], "Time:", vec_fitc_time[i-1])

    # FITC (space-time kmeans++)
    start_time = time.time()
    model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5, 
                            likelihood="gaussian",num_ind_points = 520,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="fitc")
    model.fit(y = y_train,params={"trace": True, "estimate_cov_par_index": [1,1,1,1,1,1,1,1],
                                  "init_cov_pars": cov_pars})

    pred_Linear_Model = model.predict(gp_coords_pred=coords_test,y=y_train) 
    end_time = time.time()
    vec_fitc_grid_RMSE[i-1] = np.sqrt(np.mean((y_test-pred_Linear_Model["mu"])**2))
    vec_fitc_grid_time[i-1] = end_time-start_time
    print("FITC Grid RMSE:", vec_fitc_grid_RMSE[i-1], "Time:", vec_fitc_grid_time[i-1])

    # VIF
    start_time = time.time()
    model = gpb.GPModel(gp_coords=coords_train, cov_function="space_time_gneiting", cov_fct_shape = 0.5,num_neighbors = 30,
                            likelihood="gaussian",num_ind_points = 520,ind_points_selection = "space_time_kmeans++",
                            matrix_inversion_method = "cholesky", gp_approx="full_scale_vecchia_correlation_based")
    model.fit(y = y_train,params={"trace": True, "estimate_cov_par_index": [1,1,1,1,1,1,1,1],
                                  "init_cov_pars": cov_pars})

    pred_Linear_Model = model.predict(gp_coords_pred=coords_test,y=y_train) 
    end_time = time.time()
    vec_vif_RMSE[i-1] = np.sqrt(np.mean((y_test-pred_Linear_Model["mu"])**2))
    vec_vif_time[i-1] = end_time-start_time
    print("VIF RMSE:", vec_vif_RMSE[i-1], "Time:", vec_vif_time[i-1])